# S6E4 Predicting Irrigation Need

## Score: .97216

In [1]:
from pathlib import Path
import os
import time

os.environ.setdefault("LOKY_MAX_CPU_COUNT", str(os.cpu_count() or 4))

import numpy as np
import pandas as pd
from catboost import CatBoostClassifier
from lightgbm import LGBMClassifier, early_stopping, log_evaluation
from xgboost import XGBClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import balanced_accuracy_score, confusion_matrix
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.neural_network import MLPClassifier

pd.set_option("display.max_columns", 200)


In [2]:
PROJECT_ROOT = Path(".")
DATA_DIR = PROJECT_ROOT / "playground-series-s6e4"
ORIGINAL_PATH = PROJECT_ROOT / "original-data" / "irrigation_prediction.csv"
ANCHOR_SUB_PATH = PROJECT_ROOT / "score97171.csv"
TRAIN_PATH = DATA_DIR / "train.csv"
TEST_PATH = DATA_DIR / "test.csv"
SUB_PATH = PROJECT_ROOT / "submission.csv"

TARGET = "Irrigation_Need"
ID_COL = "id"
SOURCE_COL = "_source"
BASE_SEED = 42
OVERNIGHT_MODE = False
SEED_LIST = [42, 1337, 2026]
N_SPLITS = 5 if OVERNIGHT_MODE else 3
STACKER_SPLITS = 5
STACKER_C_GRID = [
    0.25,
    0.35,
    0.45,
    0.5,
    0.65,
    0.75,
    0.85,
    0.95,
    1.0,
    1.1,
    1.25,
    1.5,
    2.0,
    3.0,
    4.0,
]
TEMPERATURE_GRID = [
    0.64,
    0.68,
    0.72,
    0.76,
    0.8,
    0.84,
    0.88,
    0.92,
    0.96,
    1.0,
    1.04,
    1.08,
    1.12,
    1.16,
]
USE_ORIGINAL_DATA = True
ORIGINAL_ROW_WEIGHT = 0.35
ANCHOR_BLEND_WEIGHT = 0.06
USE_ENGINEERED_FEATURES = True
USE_FREQ_FEATURES = True
FREQ_ENCODE_COLS = [
    "Region",
    "Crop_Type",
    "Irrigation_Type",
    "Water_Source",
    "Soil_Type",
]
USE_DIGIT_FEATURES = True
DIGIT_FEATURE_NUM_COLS = [
    "Soil_pH",
    "Soil_Moisture",
    "Organic_Carbon",
    "Electrical_Conductivity",
    "Temperature_C",
    "Humidity",
    "Rainfall_mm",
    "Sunlight_Hours",
    "Wind_Speed_kmh",
    "Field_Area_hectare",
    "Previous_Irrigation_mm",
]
DIGIT_K_MIN = -3
DIGIT_K_MAX = 2
USE_NUMERIC_ROUNDING = True
DROP_CONSTANT_FEATURES_AFTER_DIGITS = True
USE_TARGET_ENCODING = True
USE_CAT2 = True
USE_HGB = True  # sklearn HGB: different tree bias vs GBDT trio; modest extra fold time
USE_HGB2 = False  # menu B: regressed LB (~.97388); leave off unless re-tuning
USE_WEIGHTED_BASE_BLEND = True
BASE_BLEND_PRIMARY_WEIGHTS = [0.28, 0.32, 0.36]
USE_LGBM_STACKER = True  # menu C: LGBM meta-learner competes with LR on OOF
STACKER_LGB_GRID = [
    {"n_estimators": 220, "learning_rate": 0.06, "num_leaves": 31},
    {"n_estimators": 400, "learning_rate": 0.045, "num_leaves": 47},
]
USE_PER_MODEL_TEMPERATURE = True
PER_MODEL_T_GRID = [0.84, 0.92, 1.0, 1.08, 1.16]
TE_N_SPLITS = 5
TE_SMOOTHING = 25.0
USE_PSEUDO_LABEL = OVERNIGHT_MODE  # pseudo phase-2 only overnight (~2x train time)
PSEUDO_CONF_MIN = 0.994
PSEUDO_MAX_PER_CLASS = 5000
PSEUDO_ROW_WEIGHT = 0.22
PSEUDO_CURRICULUM = [
    {"name": "ultra", "conf_min": 0.997, "max_per_class": 2200, "row_weight": 0.34, "agree_min": 1},
    {"name": "high", "conf_min": 0.992, "max_per_class": 5200, "row_weight": 0.20, "agree_min": 2},
]
USE_PSEUDO_CONSENSUS_FILTER = True
USE_ANCHOR_FALLBACK = False
FALLBACK_CONF_THRESHOLD = 0.60
SUBMISSION_EXTRA_BLENDS = []
REFINE_CLASS_SCALE = False
USE_CLASSWISE_STACK_BASE_BLEND = True
CLASSWISE_BLEND_GRID = [0.0, 0.15, 0.3, 0.45, 0.6, 0.75]
USE_ADAPTIVE_CLASSWISE_REFINE = True
CLASSWISE_FINE_DELTA_STEP = 0.045
CLASSWISE_FINE_DELTA_STEPS = 3
USE_ADAPTIVE_SCALE_REFINE = True
SCALE_FINE_HALF_SPAN = 0.06
SCALE_FINE_N = 7
USE_BA_CLASS_THRESHOLD_SEARCH = True
CLASS_THRESHOLD_GRID = np.arange(-0.08, 0.081, 0.01)
CLASS_THRESHOLD_TRANSFER_STRENGTH = 0.65
USE_INTERACTION_DISCOVERY = True
INTERACTION_CANDIDATE_LIMIT = 18
INTERACTION_KEEP_TOP_K = 8
USE_CV_REPEAT_PLAN = True
CV_REPEAT_PLAN = [
    {"n_splits": 3, "seeds": [42, 1337, 2026]},
    {"n_splits": 4, "seeds": [31415]},
]
USE_STABLE_BLEND_PICK = True
USE_MLP_BASE = True
MLP_HIDDEN = (192, 96)
MLP_MAX_ITER = 80


if not TRAIN_PATH.exists() or not TEST_PATH.exists():
    raise FileNotFoundError(
        f"Expected data files under {DATA_DIR.resolve()}"
    )

train_df = pd.read_csv(TRAIN_PATH)
test_df = pd.read_csv(TEST_PATH)

train_df[SOURCE_COL] = "kaggle"

if USE_ORIGINAL_DATA and ORIGINAL_PATH.exists():
    original_df = pd.read_csv(ORIGINAL_PATH)
    original_df = original_df[[c for c in train_df.columns if c not in [ID_COL, SOURCE_COL]]]
    original_df[SOURCE_COL] = "original"
    train_df = pd.concat([train_df, original_df], axis=0, ignore_index=True)
    print("original shape:", original_df.shape)
else:
    print("original data disabled or missing")

print("train shape:", train_df.shape)
print("test shape :", test_df.shape)
train_df.head()


original shape: (10000, 21)
train shape: (640000, 22)
test shape : (270000, 20)


,id,Soil_Type,Soil_pH,Soil_Moisture,Organic_Carbon,Electrical_Conductivity,Temperature_C,Humidity,Rainfall_mm,Sunlight_Hours,Wind_Speed_kmh,Crop_Type,Crop_Growth_Stage,Season,Irrigation_Type,Water_Source,Field_Area_hectare,Mulching_Used,Previous_Irrigation_mm,Region,Irrigation_Need,_source
0,0.0,Loamy,4.92,32.58,1.01,3.05,15.01,50.61,725.99,5.90,16.79,Sugarcane,Sowing,Zaid,Drip,Rainwater,0.82,No,112.16,East,Low,kaggle
1,1.0,Clay,7.08,56.61,0.44,2.00,22.92,67.86,985.66,6.98,3.39,Wheat,Vegetative,Kharif,Rainfed,River,5.27,Yes,47.16,South,Low,kaggle
2,2.0,Clay,5.69,27.71,0.81,2.83,26.97,92.22,2201.70,6.05,3.85,Rice,Vegetative,Kharif,Sprinkler,Reservoir,8.24,Yes,110.38,North,Low,kaggle
3,3.0,Sandy,5.65,13.32,1.33,0.87,13.32,61.57,1357.33,9.12,2.31,Wheat,Flowering,Kharif,Canal,River,8.32,Yes,53.85,South,Medium,kaggle
4,4.0,Clay,7.96,59.14,0.38,0.96,20.22,91.11,1538.20,6.95,13.94,Wheat,Sowing,Rabi,Canal,River,7.37,No,93.19,South,Low,kaggle


In [3]:
display(train_df[TARGET].value_counts(dropna=False))
print("\nTarget distribution (%):")
display((train_df[TARGET].value_counts(normalize=True) * 100).round(2))

missing_train = train_df.isna().mean().sort_values(ascending=False)
missing_test = test_df.isna().mean().sort_values(ascending=False)

print("\nTop missing-rate features (train):")
display(missing_train.head(10))
print("\nTop missing-rate features (test):")
display(missing_test.head(10))

Irrigation_Need
Low       375781
Medium    242874
High       21345
Name: count, dtype: int64


Target distribution (%):


Irrigation_Need
Low       58.72
Medium    37.95
High       3.34
Name: proportion, dtype: float64


Top missing-rate features (train):


id                         0.015625
Soil_Type                  0.000000
Soil_pH                    0.000000
Soil_Moisture              0.000000
Organic_Carbon             0.000000
Electrical_Conductivity    0.000000
Temperature_C              0.000000
Humidity                   0.000000
Rainfall_mm                0.000000
Sunlight_Hours             0.000000
dtype: float64


Top missing-rate features (test):


id                         0.0
Soil_Type                  0.0
Soil_pH                    0.0
Soil_Moisture              0.0
Organic_Carbon             0.0
Electrical_Conductivity    0.0
Temperature_C              0.0
Humidity                   0.0
Rainfall_mm                0.0
Sunlight_Hours             0.0
dtype: float64

In [4]:



def add_engineered_features(df):
    out = df.copy()
    eps = 1e-6
    out["feat_rain_per_moisture"] = out["Rainfall_mm"] / (out["Soil_Moisture"] + eps)
    out["feat_humid_temp"] = out["Humidity"] * out["Temperature_C"] / 100.0
    out["feat_prev_per_area"] = out["Previous_Irrigation_mm"] / (out["Field_Area_hectare"] + eps)
    out["feat_sun_wind"] = out["Sunlight_Hours"] * out["Wind_Speed_kmh"] / 100.0
    out["feat_oc_ph"] = out["Organic_Carbon"] * out["Soil_pH"]
    out["feat_net_water_mm"] = out["Rainfall_mm"] - out["Previous_Irrigation_mm"]
    out["feat_rain_per_sun"] = out["Rainfall_mm"] / (out["Sunlight_Hours"] + eps)
    out["feat_drying_index"] = (
        out["Temperature_C"] * out["Wind_Speed_kmh"] / (out["Humidity"] + eps)
    )
    out["feat_ec_over_moisture"] = out["Electrical_Conductivity"] / (
        out["Soil_Moisture"] + eps
    )
    out["feat_moisture_ph"] = out["Soil_Moisture"] * out["Soil_pH"] / 100.0
    out["feat_irrig_share"] = out["Previous_Irrigation_mm"] / (
        out["Rainfall_mm"] + out["Previous_Irrigation_mm"] + eps
    )
    out["feat_log1p_rain"] = np.log1p(out["Rainfall_mm"].clip(lower=0))
    out["feat_log1p_prev"] = np.log1p(out["Previous_Irrigation_mm"].clip(lower=0))
    out["feat_temp_div_humid"] = out["Temperature_C"] / (out["Humidity"] + eps)
    out["feat_wind_div_rain"] = out["Wind_Speed_kmh"] / (out["Rainfall_mm"] + eps)
    out["feat_moisture_sqrt"] = np.sqrt(out["Soil_Moisture"].clip(lower=0))
    out["feat_ph_moisture"] = out["Soil_pH"] * out["Soil_Moisture"] / 100.0
    out["feat_vpd_proxy"] = out["Temperature_C"] * (1.0 - out["Humidity"] / 100.0)
    return out


def add_train_only_log_frequency_features(train_df, test_df, cat_cols):
    for col in cat_cols:
        if col not in train_df.columns:
            continue
        vc = train_df[col].value_counts()
        tr = train_df[col].map(vc).fillna(0).astype(np.float64)
        te = test_df[col].map(vc).fillna(0).astype(np.float64)
        train_df[f"logfreq_{col}"] = np.log1p(tr)
        test_df[f"logfreq_{col}"] = np.log1p(te)
    return train_df, test_df


def add_train_guided_numeric_rounding(train_df, test_df, num_cols):
    for col in num_cols:
        if col not in train_df.columns:
            continue
        tr = train_df[col].astype(np.float64)
        M = float(np.nanmax(np.abs(tr.values)))
        if not np.isfinite(M):
            continue
        if M < 10:
            decimals = 3
        elif M < 100:
            decimals = 2
        else:
            decimals = 1
        train_df[col] = tr.round(decimals)
        test_df[col] = test_df[col].astype(np.float64).round(decimals)
    return train_df, test_df


def add_digit_features(train_df, test_df, num_cols, k_min=-3, k_max=2):
    for col in num_cols:
        if col not in train_df.columns:
            continue
        tr_vals = train_df[col].astype(np.float64).abs().values
        te_vals = test_df[col].astype(np.float64).abs().values
        for k in range(k_min, k_max + 1):
            name = f"digit_{col}_{k}"
            if k >= 0:
                tr_digit = (np.floor(tr_vals / (10.0**k)) % 10).astype(np.int8)
                te_digit = (np.floor(te_vals / (10.0**k)) % 10).astype(np.int8)
            else:
                scale = 10.0 ** (-k)
                tr_digit = (np.floor(tr_vals * scale) % 10).astype(np.int8)
                te_digit = (np.floor(te_vals * scale) % 10).astype(np.int8)
            train_df[name] = tr_digit
            test_df[name] = te_digit
    return train_df, test_df


def _fit_te_map(cat_series, y_series, classes, priors, m):
    df = pd.DataFrame({"c": cat_series.values, "y": y_series.values})
    te_map = {}
    for cval, grp in df.groupby("c", sort=False):
        n = len(grp)
        counts = grp["y"].value_counts()
        arr = np.array([counts.get(cl, 0) for cl in classes], dtype=np.float64)
        te_map[cval] = (arr + m * priors) / (n + m)
    return te_map, priors.copy()


def _apply_te_rows(series, te_map, default, n_classes):
    out = np.zeros((len(series), n_classes), dtype=np.float64)
    vals = series.values
    for i in range(len(series)):
        out[i] = te_map.get(vals[i], default)
    return out


def add_multiclass_target_encoding_oof(
    train_df, test_df, target_col, cat_cols, seed, n_splits, m
):
    y = train_df[target_col]
    classes_local = sorted(y.unique().tolist())
    n_classes = len(classes_local)
    priors = (
        y.value_counts(normalize=True).reindex(classes_local).fillna(0).values.astype(np.float64)
    )
    oof_mat = np.zeros((len(train_df), len(cat_cols) * n_classes), dtype=np.float64)
    col_off = {c: i * n_classes for i, c in enumerate(cat_cols)}
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    for tr_idx, va_idx in skf.split(train_df, y):
        y_tr = y.iloc[tr_idx]
        for col in cat_cols:
            te_map, default = _fit_te_map(
                train_df[col].iloc[tr_idx], y_tr, classes_local, priors, m
            )
            block = _apply_te_rows(
                train_df[col].iloc[va_idx], te_map, default, n_classes
            )
            sl = slice(col_off[col], col_off[col] + n_classes)
            oof_mat[va_idx, sl] = block
    test_mat = np.zeros((len(test_df), len(cat_cols) * n_classes), dtype=np.float64)
    for col in cat_cols:
        te_map, default = _fit_te_map(train_df[col], y, classes_local, priors, m)
        test_mat[:, col_off[col] : col_off[col] + n_classes] = _apply_te_rows(
            test_df[col], te_map, default, n_classes
        )
    for i, col in enumerate(cat_cols):
        for k, cls in enumerate(classes_local):
            name = "te_" + str(col).replace(" ", "_") + "_" + str(cls).replace(" ", "_")
            train_df[name] = oof_mat[:, i * n_classes + k]
            test_df[name] = test_mat[:, i * n_classes + k]
    print(
        f"OOF target encoding: +{len(cat_cols) * n_classes} cols (splits={n_splits}, m={m})"
    )
    return train_df, test_df


def _oof_bal_acc_from_single_feature(series, y, seed=42, n_splits=4):
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    preds = np.empty(len(series), dtype=object)
    s = series.copy()
    is_cat = s.dtype == "object"
    if is_cat:
        s = s.fillna("<na>").astype(str)
    else:
        s = pd.to_numeric(s, errors="coerce")
        s = s.fillna(float(s.median()) if s.notna().any() else 0.0)

    def _safe_mode(z, fallback):
        vc = z.value_counts()
        return vc.index[0] if len(vc) > 0 else fallback

    for tr_idx, va_idx in skf.split(np.zeros(len(y)), y):
        s_tr = s.iloc[tr_idx]
        y_tr = y.iloc[tr_idx]
        global_mode = y_tr.value_counts().index[0]

        if is_cat:
            mode_map = y_tr.groupby(s_tr).agg(lambda z: _safe_mode(z, global_mode))
            preds[va_idx] = s.iloc[va_idx].map(mode_map).fillna(global_mode).values
        else:
            # robust numeric binning; fallback to global mode if bins collapse
            n_unique = int(s_tr.nunique())
            qn = int(max(2, min(16, n_unique)))
            if qn < 2:
                preds[va_idx] = global_mode
                continue
            s_rank = s_tr.rank(method="first")
            try:
                train_bins = pd.qcut(s_rank, q=qn, duplicates="drop")
            except Exception:
                preds[va_idx] = global_mode
                continue
            if train_bins.isna().all():
                preds[va_idx] = global_mode
                continue
            bin_map = y_tr.groupby(train_bins).agg(lambda z: _safe_mode(z, global_mode))
            q_edges = np.nanquantile(s_tr.to_numpy(dtype=np.float64), np.linspace(0.0, 1.0, qn + 1))
            edges = np.unique(q_edges)
            if len(edges) < 2:
                preds[va_idx] = global_mode
                continue
            edges[0] = -np.inf
            edges[-1] = np.inf
            val_bins = pd.cut(s.iloc[va_idx], bins=edges, include_lowest=True)
            preds[va_idx] = val_bins.map(bin_map).fillna(global_mode).values

    return float(balanced_accuracy_score(y, preds))


def add_interaction_discovery_features(train_df, test_df, target_col):
    eps = 1e-6
    y = train_df[target_col]
    candidates = []

    def add_num(name, tr, te):
        candidates.append((name, pd.to_numeric(tr, errors="coerce"), pd.to_numeric(te, errors="coerce"), "num"))

    def add_cat(name, tr, te):
        candidates.append((name, tr.astype(str), te.astype(str), "cat"))

    add_num("ix_rain_x_moist", train_df["Rainfall_mm"] * train_df["Soil_Moisture"], test_df["Rainfall_mm"] * test_df["Soil_Moisture"])
    add_num("ix_temp_x_humid", train_df["Temperature_C"] * train_df["Humidity"], test_df["Temperature_C"] * test_df["Humidity"])
    add_num("ix_prev_x_stage", train_df["Previous_Irrigation_mm"] * train_df["Crop_Growth_Stage"].astype("category").cat.codes, test_df["Previous_Irrigation_mm"] * test_df["Crop_Growth_Stage"].astype("category").cat.codes)
    add_num("ix_moisture_over_temp", train_df["Soil_Moisture"] / (train_df["Temperature_C"] + eps), test_df["Soil_Moisture"] / (test_df["Temperature_C"] + eps))
    add_num("ix_rain_over_area", train_df["Rainfall_mm"] / (train_df["Field_Area_hectare"] + eps), test_df["Rainfall_mm"] / (test_df["Field_Area_hectare"] + eps))
    add_cat("ix_season_x_stage", train_df["Season"].astype(str) + "|" + train_df["Crop_Growth_Stage"].astype(str), test_df["Season"].astype(str) + "|" + test_df["Crop_Growth_Stage"].astype(str))
    add_cat("ix_crop_x_region", train_df["Crop_Type"].astype(str) + "|" + train_df["Region"].astype(str), test_df["Crop_Type"].astype(str) + "|" + test_df["Region"].astype(str))
    add_cat("ix_soil_x_irrig", train_df["Soil_Type"].astype(str) + "|" + train_df["Irrigation_Type"].astype(str), test_df["Soil_Type"].astype(str) + "|" + test_df["Irrigation_Type"].astype(str))

    scored = []
    for name, tr, te, kind in candidates[:INTERACTION_CANDIDATE_LIMIT]:
        sc = _oof_bal_acc_from_single_feature(tr, y, seed=BASE_SEED, n_splits=4)
        scored.append((sc, name, tr, te, kind))

    scored.sort(key=lambda x: x[0], reverse=True)
    keep = scored[:INTERACTION_KEEP_TOP_K]

    for sc, name, tr, te, kind in keep:
        if kind == "num":
            train_df[name] = tr.fillna(tr.median())
            test_df[name] = te.fillna(tr.median())
        else:
            train_df[name] = tr
            test_df[name] = te

    print(
        "Interaction discovery kept:",
        [f"{nm}({score:.4f})" for score, nm, _, _, _ in keep],
    )
    return train_df, test_df


if USE_NUMERIC_ROUNDING:
    train_df, test_df = add_train_guided_numeric_rounding(
        train_df, test_df, DIGIT_FEATURE_NUM_COLS
    )

if USE_ENGINEERED_FEATURES:
    train_df = add_engineered_features(train_df)
    test_df = add_engineered_features(test_df)

if USE_FREQ_FEATURES:
    train_df, test_df = add_train_only_log_frequency_features(
        train_df, test_df, FREQ_ENCODE_COLS
    )

if USE_DIGIT_FEATURES:
    train_df, test_df = add_digit_features(
        train_df,
        test_df,
        DIGIT_FEATURE_NUM_COLS,
        DIGIT_K_MIN,
        DIGIT_K_MAX,
    )

if USE_INTERACTION_DISCOVERY:
    train_df, test_df = add_interaction_discovery_features(
        train_df,
        test_df,
        TARGET,
    )

if DROP_CONSTANT_FEATURES_AFTER_DIGITS:
    _reserved = {TARGET, ID_COL, SOURCE_COL}
    _const = [
        c
        for c in train_df.columns
        if c not in _reserved and train_df[c].nunique(dropna=False) <= 1
    ]
    if _const:
        train_df = train_df.drop(columns=_const)
        test_df = test_df.drop(columns=[c for c in _const if c in test_df.columns])
        _preview = _const[:12]
        _more = " ..." if len(_const) > 12 else ""
        print(
            f"Dropped {len(_const)} constant columns after digit features: {_preview}{_more}"
        )

pre_cols = [c for c in train_df.columns if c not in [TARGET, ID_COL, SOURCE_COL]]
_cat_for_te = [c for c in pre_cols if train_df[c].dtype == "object"]
if USE_TARGET_ENCODING and len(_cat_for_te) > 0:
    train_df, test_df = add_multiclass_target_encoding_oof(
        train_df,
        test_df,
        TARGET,
        _cat_for_te,
        BASE_SEED,
        TE_N_SPLITS,
        TE_SMOOTHING,
    )

TRAIN_DF_BASE = train_df.copy()

y = train_df[TARGET].copy()
source_flag = train_df[SOURCE_COL].copy()
X = train_df.drop(columns=[TARGET]).copy()
X_test = test_df.copy()

feature_cols = [c for c in X.columns if c not in [ID_COL, SOURCE_COL]]
X = X[feature_cols]
X_test = X_test[feature_cols]

cat_cols = [c for c in feature_cols if X[c].dtype == "object"]
num_cols = [c for c in feature_cols if c not in cat_cols]

print(f"Total features: {len(feature_cols)}")
print(f"Categorical   : {len(cat_cols)} -> {cat_cols}")
print(f"Numerical     : {len(num_cols)}")
print("Source distribution:")
display(source_flag.value_counts())


C:\Users\ol1v3_7dwns5u\AppData\Local\Temp\ipykernel_32832\2379175685.py:180: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  bin_map = y_tr.groupby(train_bins).agg(lambda z: _safe_mode(z, global_mode))
C:\Users\ol1v3_7dwns5u\AppData\Local\Temp\ipykernel_32832\2379175685.py:180: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  bin_map = y_tr.groupby(train_bins).agg(lambda z: _safe_mode(z, global_mode))
C:\Users\ol1v3_7dwns5u\AppData\Local\Temp\ipykernel_32832\2379175685.py:180: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False t

Interaction discovery kept: ['ix_season_x_stage(0.5161)', 'ix_rain_x_moist(0.3333)', 'ix_temp_x_humid(0.3333)', 'ix_prev_x_stage(0.3333)', 'ix_moisture_over_temp(0.3333)', 'ix_rain_over_area(0.3333)', 'ix_crop_x_region(0.3333)', 'ix_soil_x_irrig(0.3333)']
Dropped 13 constant columns after digit features: ['digit_Soil_pH_1', 'digit_Soil_pH_2', 'digit_Soil_Moisture_2', 'digit_Organic_Carbon_-3', 'digit_Organic_Carbon_1', 'digit_Organic_Carbon_2', 'digit_Electrical_Conductivity_1', 'digit_Electrical_Conductivity_2', 'digit_Temperature_C_2', 'digit_Humidity_2', 'digit_Sunlight_Hours_2', 'digit_Wind_Speed_kmh_2'] ...


C:\Users\ol1v3_7dwns5u\AppData\Local\Temp\ipykernel_32832\2379175685.py:134: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  test_df[name] = test_mat[:, i * n_classes + k]
C:\Users\ol1v3_7dwns5u\AppData\Local\Temp\ipykernel_32832\2379175685.py:133: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  train_df[name] = oof_mat[:, i * n_classes + k]
C:\Users\ol1v3_7dwns5u\AppData\Local\Temp\ipykernel_32832\2379175685.py:134: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times,

OOF target encoding: +33 cols (splits=5, m=25.0)
Total features: 136
Categorical   : 11 -> ['Soil_Type', 'Crop_Type', 'Crop_Growth_Stage', 'Season', 'Irrigation_Type', 'Water_Source', 'Mulching_Used', 'Region', 'ix_season_x_stage', 'ix_crop_x_region', 'ix_soil_x_irrig']
Numerical     : 125
Source distribution:


_source
kaggle      630000
original     10000
Name: count, dtype: int64

In [5]:
classes = sorted(TRAIN_DF_BASE[TARGET].unique().tolist())
class_to_idx = {label: idx for idx, label in enumerate(classes)}
idx_to_class = {v: k for k, v in class_to_idx.items()}

if OVERNIGHT_MODE:
    cat_iters, cat_lr, cat_depth, cat_es, cat_verbose = 1200, 0.04, 8, 120, 200
    lgb_est, lgb_lr, lgb_leaves, lgb_es = 1200, 0.035, 96, 120
    xgb_est, xgb_lr, xgb_depth, xgb_es = 1200, 0.035, 8, 120
    cat2_iters, cat2_lr, cat2_depth, cat2_es, cat2_verbose = 900, 0.05, 6, 100, 150
else:
    cat_iters, cat_lr, cat_depth, cat_es, cat_verbose = 700, 0.06, 7, 80, 100
    lgb_est, lgb_lr, lgb_leaves, lgb_es = 800, 0.04, 63, 80
    xgb_est, xgb_lr, xgb_depth, xgb_es = 800, 0.04, 7, 80
    cat2_iters, cat2_lr, cat2_depth, cat2_es, cat2_verbose = 500, 0.08, 6, 80, 100



def _build_hgb_numeric_matrix(X_fit, X_parts, feature_cols, cat_cols):
    """Ordinal-encode cats using X_fit only; map unknowns to len(categories)."""
    blocks_fit = []
    blocks_parts = [[] for _ in X_parts]
    cat_idx = []
    col_at = 0
    for c in feature_cols:
        if c in cat_cols:
            vc = X_fit[c].astype(str)
            uniq = sorted(vc.unique())
            code = {u: j for j, u in enumerate(uniq)}
            unk = float(len(uniq))

            def enc(X_any):
                return (
                    X_any[c]
                    .astype(str)
                    .map(lambda v, _code=code, _unk=unk: _code.get(v, _unk))
                    .astype(np.float64)
                    .to_numpy()
                    .reshape(-1, 1)
                )

            blocks_fit.append(enc(X_fit))
            for i, xp in enumerate(X_parts):
                blocks_parts[i].append(enc(xp))
            cat_idx.append(col_at)
            col_at += 1
        else:

            def num(X_any, _c=c):
                return (
                    pd.to_numeric(X_any[_c], errors="coerce")
                    .fillna(0.0)
                    .astype(np.float64)
                    .to_numpy()
                    .reshape(-1, 1)
                )

            blocks_fit.append(num(X_fit))
            for i, xp in enumerate(X_parts):
                blocks_parts[i].append(num(xp))
            col_at += 1
    X_fit_mat = np.hstack(blocks_fit)
    X_part_mats = [np.hstack(b) for b in blocks_parts]
    return X_fit_mat, X_part_mats, np.array(cat_idx, dtype=np.int64)


def build_meta_features(*parts):
    eps = 1e-15
    parts = tuple(parts)
    blocks = [np.hstack(parts)]
    for p in parts:
        blocks.append(np.log(np.clip(p, eps, 1.0)))
    for p in parts:
        ent = -(p * np.log(np.clip(p, eps, 1.0))).sum(axis=1, keepdims=True)
        blocks.append(ent)
    for p in parts:
        blocks.append(np.max(p, axis=1, keepdims=True))
    for p in parts:
        ps = np.sort(p, axis=1)
        margin = (ps[:, -1] - ps[:, -2]).reshape(-1, 1)
        blocks.append(margin)
    stacked = np.stack([np.asarray(p, dtype=np.float64) for p in parts], axis=0)
    blocks.append(stacked.std(axis=0))
    blocks.append(np.ptp(stacked, axis=0))
    return np.hstack(blocks)


def build_pseudo_augmented_train(train_df_base, test_df, test_probs):
    rng = np.random.RandomState(BASE_SEED + 99)
    conf = test_probs.max(axis=1)
    pred_i = np.argmax(test_probs, axis=1)
    keep_idx = []
    for cls_idx in range(len(classes)):
        mask = (conf >= PSEUDO_CONF_MIN) & (pred_i == cls_idx)
        ix = np.where(mask)[0]
        if len(ix) > PSEUDO_MAX_PER_CLASS:
            ix = rng.choice(ix, PSEUDO_MAX_PER_CLASS, replace=False)
        keep_idx.extend(ix.tolist())
    keep_idx = np.unique(np.array(keep_idx, dtype=int))
    pseudo = test_df.iloc[keep_idx].copy()
    pseudo[TARGET] = [idx_to_class[pred_i[i]] for i in keep_idx]
    pseudo[SOURCE_COL] = "pseudo"
    out = pd.concat([train_df_base, pseudo], axis=0, ignore_index=True)
    print(
        f"Pseudo-label: added {len(pseudo)} rows "
        f"(conf>={PSEUDO_CONF_MIN}, cap={PSEUDO_MAX_PER_CLASS}/class)"
    )
    return out


def temper_probs(p, T):
    z = np.log(np.clip(p, 1e-15, 1.0))
    zs = z / T
    ex = np.exp(zs - zs.max(axis=1, keepdims=True))
    return ex / ex.sum(axis=1, keepdims=True)


def get_cv_schedule():
    if USE_CV_REPEAT_PLAN:
        schedule = []
        for blk in CV_REPEAT_PLAN:
            ns = int(blk["n_splits"])
            for sd in blk["seeds"]:
                schedule.append((int(sd), ns))
        return schedule
    return [(int(sd), int(N_SPLITS)) for sd in SEED_LIST]


def optimize_class_threshold_offsets(proba, y_true, classes_local):
    offsets = np.zeros(len(classes_local), dtype=np.float64)
    best_score = balanced_accuracy_score(y_true, [classes_local[i] for i in np.argmax(proba, axis=1)])
    for _ in range(2):
        improved = False
        for ci in range(len(classes_local)):
            base = offsets[ci]
            best_local = best_score
            best_val = base
            for d in CLASS_THRESHOLD_GRID:
                trial = offsets.copy()
                trial[ci] = base + float(d)
                pred = np.argmax(proba + trial, axis=1)
                sc = balanced_accuracy_score(y_true, [classes_local[i] for i in pred])
                if sc > best_local:
                    best_local = sc
                    best_val = trial[ci]
            if best_local > best_score:
                offsets[ci] = best_val
                best_score = best_local
                improved = True
        if not improved:
            break
    return offsets, best_score


def build_pseudo_augmented_train_curriculum(train_df_base, test_df_in, proba_dict):
    rng = np.random.RandomState(BASE_SEED + 199)
    keep_frames = []
    for cfg in PSEUDO_CURRICULUM:
        probs = proba_dict["stack"]
        pred_i = np.argmax(probs, axis=1)
        conf = probs.max(axis=1)
        if USE_PSEUDO_CONSENSUS_FILTER:
            agree = np.zeros(len(test_df_in), dtype=np.int64)
            for key in ["stack", "cat", "lgb", "xgb"]:
                if key in proba_dict:
                    agree += (np.argmax(proba_dict[key], axis=1) == pred_i).astype(np.int64)
            agree_need = int(cfg.get("agree_min", 1))
        else:
            agree = np.ones(len(test_df_in), dtype=np.int64)
            agree_need = 1
        cur_keep = []
        for cls_idx in range(len(classes)):
            mask = (pred_i == cls_idx) & (conf >= float(cfg["conf_min"])) & (agree >= agree_need)
            ix = np.where(mask)[0]
            cap = int(cfg["max_per_class"])
            if len(ix) > cap:
                ix = rng.choice(ix, cap, replace=False)
            cur_keep.extend(ix.tolist())
        cur_keep = np.unique(np.array(cur_keep, dtype=np.int64))
        if len(cur_keep) == 0:
            continue
        pseudo = test_df_in.iloc[cur_keep].copy()
        pseudo[TARGET] = [idx_to_class[pred_i[i]] for i in cur_keep]
        pseudo[SOURCE_COL] = f"pseudo_{cfg['name']}"
        pseudo["_pseudo_weight"] = float(cfg["row_weight"])
        keep_frames.append(pseudo)
        print(f"Pseudo stage {cfg['name']}: added {len(pseudo)} rows (conf>={cfg['conf_min']}, agree>={agree_need})")
    if len(keep_frames) == 0:
        print("Pseudo curriculum produced 0 rows; skipping phase2.")
        return train_df_base.copy(), {}
    all_pseudo = pd.concat(keep_frames, axis=0, ignore_index=True)
    all_pseudo = all_pseudo.drop_duplicates(subset=[ID_COL], keep="first")
    out = pd.concat([train_df_base, all_pseudo], axis=0, ignore_index=True)
    weight_map = {
        row[ID_COL]: row.get("_pseudo_weight", PSEUDO_ROW_WEIGHT)
        for _, row in all_pseudo[[ID_COL, "_pseudo_weight"]].iterrows()
    }
    return out, weight_map


def run_training_phase(train_df_run, phase_name, pseudo_weight_map=None):
    run_start = time.perf_counter()
    print(f"\n{'='*20} {phase_name} {'='*20}")
    print(
        f"Config: OVERNIGHT_MODE={OVERNIGHT_MODE} | seeds={len(SEED_LIST)} | "
        f"n_splits={N_SPLITS} | stacker_splits={STACKER_SPLITS} | USE_CAT2={USE_CAT2} | "
        f"USE_HGB={USE_HGB} | USE_HGB2={USE_HGB2} | "
        f"USE_LGBM_STACKER={USE_LGBM_STACKER} | "
        f"C_grid={len(STACKER_C_GRID)} | "
        f"LGB_stack_grid={len(STACKER_LGB_GRID) if USE_LGBM_STACKER else 0} | "
        f"T_grid={len(TEMPERATURE_GRID)}"
    )
    if USE_CV_REPEAT_PLAN:
        print(f"CV repeat schedule: {CV_REPEAT_PLAN}")

    y = train_df_run[TARGET].copy()
    source_flag = train_df_run[SOURCE_COL].copy()
    X = train_df_run.drop(columns=[TARGET]).copy()
    X_test = test_df.copy()
    feature_cols = [c for c in X.columns if c not in [ID_COL, SOURCE_COL]]
    X = X[feature_cols]
    X_test = X_test[feature_cols]
    cat_cols = [c for c in feature_cols if X[c].dtype == "object"]

    if USE_HGB2 and not USE_HGB:
        raise ValueError("USE_HGB2 requires USE_HGB=True")

    y_idx = y.map(class_to_idx).astype(int)
    class_counts = y.value_counts()
    class_weight_label = {
        cls: len(y) / (len(classes) * class_counts[cls])
        for cls in classes
    }
    class_weight_idx = {
        class_to_idx[cls]: weight
        for cls, weight in class_weight_label.items()
    }
    source_weight_map = {
        "kaggle": 1.0,
        "original": ORIGINAL_ROW_WEIGHT,
        "pseudo": PSEUDO_ROW_WEIGHT,
    }
    if pseudo_weight_map is None:
        pseudo_weight_map = {}

    oof_cat = np.zeros((len(X), len(classes)), dtype=np.float64)
    oof_cat2 = np.zeros((len(X), len(classes)), dtype=np.float64)
    oof_lgb = np.zeros((len(X), len(classes)), dtype=np.float64)
    oof_xgb = np.zeros((len(X), len(classes)), dtype=np.float64)

    oof_hgb = np.zeros((len(X), len(classes)), dtype=np.float64)
    oof_hgb2 = np.zeros((len(X), len(classes)), dtype=np.float64)
    oof_mlp = np.zeros((len(X), len(classes)), dtype=np.float64)

    test_cat = np.zeros((len(X_test), len(classes)), dtype=np.float64)
    test_cat2 = np.zeros((len(X_test), len(classes)), dtype=np.float64)
    test_lgb = np.zeros((len(X_test), len(classes)), dtype=np.float64)
    test_xgb = np.zeros((len(X_test), len(classes)), dtype=np.float64)

    test_hgb = np.zeros((len(X_test), len(classes)), dtype=np.float64)
    test_hgb2 = np.zeros((len(X_test), len(classes)), dtype=np.float64)
    test_mlp = np.zeros((len(X_test), len(classes)), dtype=np.float64)

    fold_scores = []

    cv_schedule = get_cv_schedule()
    total_rounds = len(cv_schedule)

    for seed_idx, (seed, n_splits_local) in enumerate(cv_schedule, start=1):
        seed_start = time.perf_counter()
        print(f"\n######## CV round {seed_idx}/{total_rounds} (seed={seed}, splits={n_splits_local}) ########")

        cv = StratifiedKFold(n_splits=n_splits_local, shuffle=True, random_state=seed)

        seed_test_cat = np.zeros((len(X_test), len(classes)), dtype=np.float64)
        seed_test_cat2 = np.zeros((len(X_test), len(classes)), dtype=np.float64)
        seed_test_lgb = np.zeros((len(X_test), len(classes)), dtype=np.float64)
        seed_test_xgb = np.zeros((len(X_test), len(classes)), dtype=np.float64)

        seed_test_hgb = np.zeros((len(X_test), len(classes)), dtype=np.float64)
        seed_test_hgb2 = np.zeros((len(X_test), len(classes)), dtype=np.float64)
        seed_test_mlp = np.zeros((len(X_test), len(classes)), dtype=np.float64)

        for fold, (train_idx, valid_idx) in enumerate(cv.split(X, y), start=1):
            fold_start = time.perf_counter()
            print(f"\n=== Seed {seed} | Fold {fold}/{n_splits_local} started ===")

            X_train, X_valid = X.iloc[train_idx].copy(), X.iloc[valid_idx].copy()
            y_train, y_valid = y.iloc[train_idx], y.iloc[valid_idx]
            y_train_idx, y_valid_idx = y_idx.iloc[train_idx], y_idx.iloc[valid_idx]
            source_train = source_flag.iloc[train_idx]

            for col in cat_cols:
                X_train[col] = X_train[col].astype("category")
                X_valid[col] = X_valid[col].astype("category")

            X_test_fold = X_test.copy()
            for col in cat_cols:
                X_test_fold[col] = X_test_fold[col].astype("category")

            source_w = source_train.map(source_weight_map).fillna(PSEUDO_ROW_WEIGHT).values.astype(np.float64)
            if len(pseudo_weight_map) > 0:
                ids_tr = train_df_run.iloc[train_idx][ID_COL].values
                for ii, rid in enumerate(ids_tr):
                    if rid in pseudo_weight_map:
                        source_w[ii] = float(pseudo_weight_map[rid])
            sample_weight = y_train.map(class_weight_label).values * source_w

            cat_model = CatBoostClassifier(
                iterations=cat_iters,
                learning_rate=cat_lr,
                depth=cat_depth,
                loss_function="MultiClass",
                eval_metric="TotalF1",
                random_seed=seed + fold,
                verbose=cat_verbose,
                thread_count=-1,
                class_weights=class_weight_label,
            )

            lgb_model = LGBMClassifier(
                n_estimators=lgb_est,
                learning_rate=lgb_lr,
                num_leaves=lgb_leaves,
                min_child_samples=40,
                subsample=0.9,
                colsample_bytree=0.9,
                reg_lambda=1.0,
                reg_alpha=0.0,
                objective="multiclass",
                random_state=seed + fold,
                class_weight=class_weight_idx,
                verbosity=-1,
            )

            xgb_model = XGBClassifier(
                n_estimators=xgb_est,
                learning_rate=xgb_lr,
                max_depth=xgb_depth,
                min_child_weight=3,
                subsample=0.9,
                colsample_bytree=0.9,
                reg_lambda=1.0,
                objective="multi:softprob",
                num_class=len(classes),
                eval_metric="mlogloss",
                tree_method="hist",
                enable_categorical=True,
                random_state=seed + fold,
                n_jobs=-1,
            )

            print(f"Seed {seed} Fold {fold}: training CatBoost...")
            cat_model.fit(
                X_train,
                y_train,
                sample_weight=sample_weight,
                cat_features=cat_cols,
                eval_set=(X_valid, y_valid),
                use_best_model=True,
                early_stopping_rounds=cat_es,
            )

            if USE_CAT2:
                cat2_model = CatBoostClassifier(
                    iterations=cat2_iters,
                    learning_rate=cat2_lr,
                    depth=cat2_depth,
                    loss_function="MultiClass",
                    eval_metric="TotalF1",
                    random_seed=seed + fold + 777,
                    verbose=cat2_verbose,
                    thread_count=-1,
                    class_weights=class_weight_label,
                )
                print(f"Seed {seed} Fold {fold}: training CatBoost-2...")
                cat2_model.fit(
                    X_train,
                    y_train,
                    sample_weight=sample_weight,
                    cat_features=cat_cols,
                    eval_set=(X_valid, y_valid),
                    use_best_model=True,
                    early_stopping_rounds=cat2_es,
                )

            print(f"Seed {seed} Fold {fold}: training LightGBM...")
            lgb_model.fit(
                X_train,
                y_train_idx,
                sample_weight=sample_weight,
                eval_set=[(X_valid, y_valid_idx)],
                eval_metric="multi_logloss",
                callbacks=[early_stopping(stopping_rounds=lgb_es), log_evaluation(period=0)],
            )

            print(f"Seed {seed} Fold {fold}: training XGBoost...")
            xgb_model.fit(
                X_train,
                y_train_idx,
                eval_set=[(X_valid, y_valid_idx)],
                sample_weight=sample_weight,
                verbose=False,
            )

            if USE_MLP_BASE:
                print(f"Seed {seed} Fold {fold}: training MLP base...")

                num_cols_local = [c for c in feature_cols if c not in cat_cols]
                X_tr_num = (
                    X_train[num_cols_local]
                    .apply(pd.to_numeric, errors="coerce")
                    .fillna(0.0)
                    .astype(np.float32)
                )
                X_va_num = (
                    X_valid[num_cols_local]
                    .apply(pd.to_numeric, errors="coerce")
                    .fillna(0.0)
                    .astype(np.float32)
                )
                X_te_num = (
                    X_test_fold[num_cols_local]
                    .apply(pd.to_numeric, errors="coerce")
                    .fillna(0.0)
                    .astype(np.float32)
                )

                if len(cat_cols) > 0:
                    X_tr_cat = pd.get_dummies(X_train[cat_cols].astype(str), drop_first=False)
                    X_va_cat = pd.get_dummies(X_valid[cat_cols].astype(str), drop_first=False)
                    X_te_cat = pd.get_dummies(X_test_fold[cat_cols].astype(str), drop_first=False)
                    X_va_cat = X_va_cat.reindex(columns=X_tr_cat.columns, fill_value=0)
                    X_te_cat = X_te_cat.reindex(columns=X_tr_cat.columns, fill_value=0)
                    X_tr_mlp = np.hstack([X_tr_num.values, X_tr_cat.values.astype(np.float32)])
                    X_va_mlp = np.hstack([X_va_num.values, X_va_cat.values.astype(np.float32)])
                    X_te_mlp = np.hstack([X_te_num.values, X_te_cat.values.astype(np.float32)])
                else:
                    X_tr_mlp = X_tr_num.values
                    X_va_mlp = X_va_num.values
                    X_te_mlp = X_te_num.values

                mlp_model = MLPClassifier(
                    hidden_layer_sizes=MLP_HIDDEN,
                    activation="relu",
                    alpha=3e-4,
                    learning_rate_init=6e-4,
                    max_iter=MLP_MAX_ITER,
                    random_state=seed + fold + 5150,
                    early_stopping=True,
                    validation_fraction=0.08,
                )
                mlp_model.fit(X_tr_mlp, y_train_idx)

            if USE_HGB:
                print(f"Seed {seed} Fold {fold}: training HistGradientBoosting...")
                X_fit_hgb, (X_va_hgb, X_te_hgb), cat_feat_idx = (
                    _build_hgb_numeric_matrix(
                        X_train, (X_valid, X_test_fold), feature_cols, cat_cols
                    )
                )
                hgb_model = HistGradientBoostingClassifier(
                    max_iter=480,
                    learning_rate=0.06,
                    max_depth=10,
                    max_leaf_nodes=96,
                    min_samples_leaf=40,
                    l2_regularization=1.0,
                    early_stopping=False,
                    random_state=seed + fold + 4242,
                    categorical_features=(
                        cat_feat_idx if len(cat_feat_idx) > 0 else None
                    ),
                )
                hgb_model.fit(X_fit_hgb, y_train_idx, sample_weight=sample_weight)
                if USE_HGB2:
                    hgb2_model = HistGradientBoostingClassifier(
                        max_iter=320,
                        learning_rate=0.09,
                        max_depth=8,
                        max_leaf_nodes=64,
                        min_samples_leaf=20,
                        l2_regularization=0.5,
                        early_stopping=False,
                        random_state=seed + fold + 9191,
                        categorical_features=(
                            cat_feat_idx if len(cat_feat_idx) > 0 else None
                        ),
                    )
                    print(
                        f"Seed {seed} Fold {fold}: training HistGradientBoosting-2..."
                    )
                    hgb2_model.fit(
                        X_fit_hgb, y_train_idx, sample_weight=sample_weight
                    )
            print(f"Seed {seed} Fold {fold}: model training complete")

            cat_valid = cat_model.predict_proba(X_valid)
            if USE_CAT2:
                cat2_valid = cat2_model.predict_proba(X_valid)
            else:
                cat2_valid = None
            lgb_valid = lgb_model.predict_proba(X_valid)
            xgb_valid = xgb_model.predict_proba(X_valid)
            if USE_MLP_BASE:
                mlp_valid = mlp_model.predict_proba(X_va_mlp)

            oof_cat[valid_idx] += cat_valid / total_rounds
            if USE_CAT2:
                oof_cat2[valid_idx] += cat2_valid / total_rounds
            oof_lgb[valid_idx] += lgb_valid / total_rounds
            oof_xgb[valid_idx] += xgb_valid / total_rounds
            if USE_MLP_BASE:
                oof_mlp[valid_idx] += mlp_valid / total_rounds
            if USE_HGB:
                hgb_valid = hgb_model.predict_proba(X_va_hgb)
                oof_hgb[valid_idx] += hgb_valid / total_rounds
                if USE_HGB2:
                    hgb2_valid = hgb2_model.predict_proba(X_va_hgb)
                    oof_hgb2[valid_idx] += hgb2_valid / total_rounds

            cat_pred = np.argmax(cat_valid, axis=1)
            lgb_pred = np.argmax(lgb_valid, axis=1)
            xgb_pred = np.argmax(xgb_valid, axis=1)

            cat_score = balanced_accuracy_score(y_valid, [idx_to_class[i] for i in cat_pred])
            lgb_score = balanced_accuracy_score(y_valid, [idx_to_class[i] for i in lgb_pred])
            xgb_score = balanced_accuracy_score(y_valid, [idx_to_class[i] for i in xgb_pred])
            if USE_CAT2:
                cat2_pred = np.argmax(cat2_valid, axis=1)
                cat2_score = balanced_accuracy_score(
                    y_valid, [idx_to_class[i] for i in cat2_pred]
                )
                if USE_HGB and USE_HGB2:
                    hgb_pred = np.argmax(hgb_valid, axis=1)
                    hgb_score = balanced_accuracy_score(
                        y_valid, [idx_to_class[i] for i in hgb_pred]
                    )
                    hgb2_pred = np.argmax(hgb2_valid, axis=1)
                    hgb2_score = balanced_accuracy_score(
                        y_valid, [idx_to_class[i] for i in hgb2_pred]
                    )
                    print(
                        f"Seed {seed} Fold {fold}: cat={cat_score:.6f} cat2={cat2_score:.6f} "
                        f"lgb={lgb_score:.6f} xgb={xgb_score:.6f} "
                        f"hgb={hgb_score:.6f} hgb2={hgb2_score:.6f}"
                    )
                elif USE_HGB:
                    hgb_pred = np.argmax(hgb_valid, axis=1)
                    hgb_score = balanced_accuracy_score(
                        y_valid, [idx_to_class[i] for i in hgb_pred]
                    )
                    print(
                        f"Seed {seed} Fold {fold}: cat={cat_score:.6f} cat2={cat2_score:.6f} "
                        f"lgb={lgb_score:.6f} xgb={xgb_score:.6f} hgb={hgb_score:.6f}"
                    )
                else:
                    print(
                        f"Seed {seed} Fold {fold}: cat={cat_score:.6f} cat2={cat2_score:.6f} "
                        f"lgb={lgb_score:.6f} xgb={xgb_score:.6f}"
                    )
            else:
                if USE_HGB and USE_HGB2:
                    hgb_pred = np.argmax(hgb_valid, axis=1)
                    hgb_score = balanced_accuracy_score(
                        y_valid, [idx_to_class[i] for i in hgb_pred]
                    )
                    hgb2_pred = np.argmax(hgb2_valid, axis=1)
                    hgb2_score = balanced_accuracy_score(
                        y_valid, [idx_to_class[i] for i in hgb2_pred]
                    )
                    print(
                        f"Seed {seed} Fold {fold}: cat={cat_score:.6f} "
                        f"lgb={lgb_score:.6f} xgb={xgb_score:.6f} "
                        f"hgb={hgb_score:.6f} hgb2={hgb2_score:.6f}"
                    )
                elif USE_HGB:
                    hgb_pred = np.argmax(hgb_valid, axis=1)
                    hgb_score = balanced_accuracy_score(
                        y_valid, [idx_to_class[i] for i in hgb_pred]
                    )
                    print(
                        f"Seed {seed} Fold {fold}: cat={cat_score:.6f} "
                        f"lgb={lgb_score:.6f} xgb={xgb_score:.6f} hgb={hgb_score:.6f}"
                    )
                else:
                    print(
                        f"Seed {seed} Fold {fold}: cat={cat_score:.6f} "
                        f"lgb={lgb_score:.6f} xgb={xgb_score:.6f}"
                    )

            if USE_CAT2 and USE_HGB and USE_HGB2:
                blend_equal = (
                    cat_valid
                    + cat2_valid
                    + lgb_valid
                    + xgb_valid
                    + hgb_valid
                    + hgb2_valid
                ) / 6.0
            elif USE_CAT2 and USE_HGB:
                blend_equal = (
                    cat_valid + cat2_valid + lgb_valid + xgb_valid + hgb_valid
                ) / 5.0
            elif USE_CAT2:
                blend_equal = (cat_valid + cat2_valid + lgb_valid + xgb_valid) / 4.0
            elif USE_HGB and USE_HGB2:
                blend_equal = (
                    cat_valid + lgb_valid + xgb_valid + hgb_valid + hgb2_valid
                ) / 5.0
            elif USE_HGB:
                blend_equal = (cat_valid + lgb_valid + xgb_valid + hgb_valid) / 4.0
            else:
                blend_equal = (cat_valid + lgb_valid + xgb_valid) / 3.0
            blend_pred = np.argmax(blend_equal, axis=1)
            fold_score = balanced_accuracy_score(y_valid, [idx_to_class[i] for i in blend_pred])
            fold_scores.append(fold_score)
            fold_minutes = (time.perf_counter() - fold_start) / 60.0
            print(f"Seed {seed} Fold {fold}: equal-blend balanced_accuracy = {fold_score:.6f}")
            print(f"Seed {seed} Fold {fold}: completed in {fold_minutes:.2f} minutes")

            seed_test_cat += cat_model.predict_proba(X_test_fold)
            if USE_CAT2:
                seed_test_cat2 += cat2_model.predict_proba(X_test_fold)
            seed_test_lgb += lgb_model.predict_proba(X_test_fold)
            seed_test_xgb += xgb_model.predict_proba(X_test_fold)
            if USE_MLP_BASE:
                seed_test_mlp += mlp_model.predict_proba(X_te_mlp)
            if USE_HGB:
                seed_test_hgb += hgb_model.predict_proba(X_te_hgb)
                if USE_HGB2:
                    seed_test_hgb2 += hgb2_model.predict_proba(X_te_hgb)

        test_cat += seed_test_cat / n_splits_local
        test_cat2 += seed_test_cat2 / n_splits_local
        test_lgb += seed_test_lgb / n_splits_local
        test_xgb += seed_test_xgb / n_splits_local
        if USE_MLP_BASE:
            test_mlp += seed_test_mlp / n_splits_local
        if USE_HGB:
            test_hgb += seed_test_hgb / n_splits_local
            if USE_HGB2:
                test_hgb2 += seed_test_hgb2 / n_splits_local

        print(
            f"Seed {seed} finished in {(time.perf_counter() - seed_start) / 60.0:.2f} minutes"
        )

    test_cat /= total_rounds
    test_cat2 /= total_rounds
    test_lgb /= total_rounds
    test_xgb /= total_rounds
    if USE_MLP_BASE:
        test_mlp /= total_rounds
    if USE_HGB:
        test_hgb /= total_rounds
        if USE_HGB2:
            test_hgb2 /= total_rounds

    if USE_PER_MODEL_TEMPERATURE:
        temp_parts = [
            ("cat", oof_cat, test_cat),
            ("lgb", oof_lgb, test_lgb),
            ("xgb", oof_xgb, test_xgb),
        ]
        if USE_MLP_BASE:
            temp_parts.append(("mlp", oof_mlp, test_mlp))
        if USE_CAT2:
            temp_parts.append(("cat2", oof_cat2, test_cat2))
        if USE_HGB:
            temp_parts.append(("hgb", oof_hgb, test_hgb))
        if USE_HGB2:
            temp_parts.append(("hgb2", oof_hgb2, test_hgb2))

        temp_log = {}
        for name, part_oof, part_test in temp_parts:
            best_t_local = 1.0
            best_s_local = -1.0
            for t_local in PER_MODEL_T_GRID:
                p_local = temper_probs(part_oof, t_local)
                pred_local = [idx_to_class[i] for i in np.argmax(p_local, axis=1)]
                s_local = balanced_accuracy_score(y, pred_local)
                if s_local > best_s_local:
                    best_s_local = s_local
                    best_t_local = t_local
            part_oof[:, :] = temper_probs(part_oof, best_t_local)
            part_test[:, :] = temper_probs(part_test, best_t_local)
            temp_log[name] = (best_t_local, best_s_local)

        print("Per-model temperature picks:")
        for name in sorted(temp_log.keys()):
            t_local, s_local = temp_log[name]
            print(f"  {name}: T={t_local} | OOF={s_local:.6f}")

    print("\nAll folds done. Building meta-features + cross-fitted OOF stacking...")

    active_oof_parts = [oof_cat, oof_lgb, oof_xgb]
    active_test_parts = [test_cat, test_lgb, test_xgb]
    if USE_CAT2:
        active_oof_parts.append(oof_cat2)
        active_test_parts.append(test_cat2)
    if USE_HGB:
        active_oof_parts.append(oof_hgb)
        active_test_parts.append(test_hgb)
    if USE_HGB2:
        active_oof_parts.append(oof_hgb2)
        active_test_parts.append(test_hgb2)
    if USE_MLP_BASE:
        active_oof_parts.append(oof_mlp)
        active_test_parts.append(test_mlp)

    meta_oof = build_meta_features(*active_oof_parts)
    meta_test = build_meta_features(*active_test_parts)
    print(f"Meta feature columns: {meta_oof.shape[1]}")

    best_stack_score = -1.0
    best_stack_c = None
    best_stack_kind = "lr"
    best_stack_label = None
    best_stack_proba_oof = None
    best_stack_proba_test = None

    stack_cv = StratifiedKFold(n_splits=STACKER_SPLITS, shuffle=True, random_state=BASE_SEED)

    for c in STACKER_C_GRID:
        print(f"Stacking search: cross-fitting LogisticRegression C={c}")

        cv_meta_oof = np.zeros((len(meta_oof), len(classes)), dtype=np.float64)
        cv_meta_test = np.zeros((len(meta_test), len(classes)), dtype=np.float64)

        for stack_fold, (st_tr_idx, st_va_idx) in enumerate(stack_cv.split(meta_oof, y_idx), start=1):
            X_st_tr, X_st_va = meta_oof[st_tr_idx], meta_oof[st_va_idx]
            y_st_tr = y_idx.iloc[st_tr_idx]

            stacker = Pipeline(
                [
                    ("scaler", StandardScaler()),
                    (
                        "clf",
                        LogisticRegression(
                            C=c,
                            max_iter=800,
                            solver="lbfgs",
                            class_weight="balanced",
                        ),
                    ),
                ]
            )
            stacker.fit(X_st_tr, y_st_tr)

            cv_meta_oof[st_va_idx] = stacker.predict_proba(X_st_va)
            cv_meta_test += stacker.predict_proba(meta_test)

            if stack_fold % 2 == 0:
                print(f"Stacking C={c}: finished meta-fold {stack_fold}/{STACKER_SPLITS}")

        cv_meta_test /= STACKER_SPLITS

        pred_oof = np.argmax(cv_meta_oof, axis=1)
        pred_oof_labels = [idx_to_class[i] for i in pred_oof]
        score = balanced_accuracy_score(y, pred_oof_labels)
        print(f"Stacking C={c}: cross-fitted OOF balanced_accuracy = {score:.6f}")

        if score > best_stack_score:
            best_stack_score = score
            best_stack_kind = "lr"
            best_stack_c = c
            best_stack_label = f"lr_C={c}"
            best_stack_proba_oof = cv_meta_oof
            best_stack_proba_test = cv_meta_test

    if USE_LGBM_STACKER:
        for li, lcfg in enumerate(STACKER_LGB_GRID):
            print(
                f"Stacking search: cross-fitting LGBM stacker "
                f"({li + 1}/{len(STACKER_LGB_GRID)} "
                f"est={lcfg['n_estimators']} lr={lcfg['learning_rate']} "
                f"leaves={lcfg['num_leaves']})"
            )
            cv_meta_oof = np.zeros((len(meta_oof), len(classes)), dtype=np.float64)
            cv_meta_test = np.zeros((len(meta_test), len(classes)), dtype=np.float64)
            for stack_fold, (st_tr_idx, st_va_idx) in enumerate(
                stack_cv.split(meta_oof, y_idx), start=1
            ):
                X_st_tr, X_st_va = meta_oof[st_tr_idx], meta_oof[st_va_idx]
                y_st_tr = y_idx.iloc[st_tr_idx]

                stacker = LGBMClassifier(
                    n_estimators=int(lcfg["n_estimators"]),
                    learning_rate=float(lcfg["learning_rate"]),
                    num_leaves=int(lcfg["num_leaves"]),
                    min_child_samples=30,
                    subsample=0.9,
                    colsample_bytree=0.9,
                    reg_lambda=1.0,
                    objective="multiclass",
                    class_weight="balanced",
                    random_state=BASE_SEED + stack_fold + li * 97,
                    verbosity=-1,
                )
                stacker.fit(
                    X_st_tr,
                    y_st_tr,
                    eval_set=[(X_st_va, y_idx.iloc[st_va_idx])],
                    eval_metric="multi_logloss",
                    callbacks=[
                        early_stopping(stopping_rounds=60),
                        log_evaluation(period=0),
                    ],
                )
                cv_meta_oof[st_va_idx] = stacker.predict_proba(X_st_va)
                cv_meta_test += stacker.predict_proba(meta_test)
                if stack_fold % 2 == 0:
                    print(
                        f"Stacking LGBM li={li}: finished meta-fold "
                        f"{stack_fold}/{STACKER_SPLITS}"
                    )

            cv_meta_test /= STACKER_SPLITS

            pred_oof = np.argmax(cv_meta_oof, axis=1)
            pred_oof_labels = [idx_to_class[i] for i in pred_oof]
            score = balanced_accuracy_score(y, pred_oof_labels)
            print(
                f"Stacking LGBM li={li}: cross-fitted OOF balanced_accuracy = {score:.6f}"
            )

            if score > best_stack_score:
                best_stack_score = score
                best_stack_kind = "lgb"
                best_stack_c = None
                best_stack_label = (
                    "lgb_est"
                    + str(int(lcfg["n_estimators"]))
                    + "_lr"
                    + str(float(lcfg["learning_rate"]))
                    + "_leaf"
                    + str(int(lcfg["num_leaves"]))
                )
                best_stack_proba_oof = cv_meta_oof
                best_stack_proba_test = cv_meta_test

    if best_stack_kind == "lr":
        print(
            f"Best stacker: LogisticRegression C={best_stack_c} | "
            f"OOF={best_stack_score:.6f}"
        )
    else:
        print(
            f"Best stacker: LGBM | {best_stack_label} | OOF={best_stack_score:.6f}"
        )
    print("Searching temperature scaling on stacker OOF...")

    best_T = 1.0
    best_temp_score = -1.0
    for T in TEMPERATURE_GRID:
        tp = temper_probs(best_stack_proba_oof, T)
        pred_lbl = [idx_to_class[i] for i in np.argmax(tp, axis=1)]
        sc = balanced_accuracy_score(y, pred_lbl)
        if sc > best_temp_score:
            best_temp_score = sc
            best_T = T

    print(f"Best temperature T={best_T} OOF balanced_accuracy={best_temp_score:.6f}")

    oof_after_temp = temper_probs(best_stack_proba_oof, best_T)
    test_after_temp = temper_probs(best_stack_proba_test, best_T)

    print("Starting class-bias tuning...")

    best_bias_score = -1.0
    best_bias = np.zeros(len(classes), dtype=np.float64)

    bias_checks = 0
    for b_low in np.arange(-0.06, 0.061, 0.01):
        for b_med in np.arange(-0.06, 0.061, 0.01):
            for b_high in np.arange(-0.06, 0.061, 0.01):
                bias = np.array([b_low, b_med, b_high], dtype=np.float64)
                biased_oof = oof_after_temp + bias
                pred_idx = np.argmax(biased_oof, axis=1)
                pred_labels = [idx_to_class[i] for i in pred_idx]
                score = balanced_accuracy_score(y, pred_labels)
                bias_checks += 1
                if bias_checks % 200 == 0:
                    print(f"Bias search progress: checked {bias_checks} combos")
                if score > best_bias_score:
                    best_bias_score = score
                    best_bias = bias.copy()

    oof_best = oof_after_temp + best_bias
    oof_best_pred = np.argmax(oof_best, axis=1)
    oof_best_pred = [idx_to_class[i] for i in oof_best_pred]

    test_proba = test_after_temp + best_bias

    base_parts = [
        ("cat", oof_cat, test_cat),
        ("lgb", oof_lgb, test_lgb),
        ("xgb", oof_xgb, test_xgb),
    ]
    if USE_CAT2:
        base_parts.append(("cat2", oof_cat2, test_cat2))
    if USE_HGB:
        base_parts.append(("hgb", oof_hgb, test_hgb))
    if USE_HGB2:
        base_parts.append(("hgb2", oof_hgb2, test_hgb2))
    if USE_MLP_BASE:
        base_parts.append(("mlp", oof_mlp, test_mlp))

    n_parts = len(base_parts)
    equal_w = np.full(n_parts, 1.0 / n_parts, dtype=np.float64)
    candidate_weights = [("equal", equal_w)]
    if USE_WEIGHTED_BASE_BLEND:
        for w_primary in BASE_BLEND_PRIMARY_WEIGHTS:
            if not (0.0 < w_primary < 1.0):
                continue
            if n_parts <= 1:
                continue
            w_rest = (1.0 - w_primary) / (n_parts - 1)
            if w_rest <= 0:
                continue
            for i, (name, _, _) in enumerate(base_parts):
                w = np.full(n_parts, w_rest, dtype=np.float64)
                w[i] = w_primary
                candidate_weights.append((f"{name}_primary_{w_primary:.2f}", w))

    best_base_name = "equal"
    best_base_score = -1.0
    best_base_weights = equal_w
    base_oof = None
    base_test = None
    candidate_eval = []

    for cname, w in candidate_weights:
        cand_oof = np.zeros_like(oof_cat, dtype=np.float64)
        cand_test = np.zeros_like(test_cat, dtype=np.float64)
        for wi, (_, part_oof, part_test) in zip(w, base_parts):
            cand_oof += wi * part_oof
            cand_test += wi * part_test
        cand_pred = [idx_to_class[i] for i in np.argmax(cand_oof, axis=1)]
        cand_score = balanced_accuracy_score(y, cand_pred)
        candidate_eval.append((cand_score, cname, w.copy(), cand_oof.copy(), cand_test.copy()))
        if cand_score > best_base_score:
            best_base_score = cand_score
            best_base_name = cname
            best_base_weights = w.copy()
            base_oof = cand_oof
            base_test = cand_test

    if USE_STABLE_BLEND_PICK and len(candidate_eval) >= 3:
        candidate_eval.sort(key=lambda x: x[0], reverse=True)
        top = candidate_eval[:3]
        stable_w = np.median(np.vstack([t[2] for t in top]), axis=0)
        stable_w = stable_w / np.clip(stable_w.sum(), 1e-12, None)
        stable_oof = np.zeros_like(oof_cat, dtype=np.float64)
        stable_test = np.zeros_like(test_cat, dtype=np.float64)
        for wi, (_, part_oof, part_test) in zip(stable_w, base_parts):
            stable_oof += wi * part_oof
            stable_test += wi * part_test
        stable_score = balanced_accuracy_score(y, [idx_to_class[i] for i in np.argmax(stable_oof, axis=1)])
        if stable_score > best_base_score:
            best_base_score = stable_score
            best_base_name = "stable_top3_median"
            best_base_weights = stable_w
            base_oof = stable_oof
            base_test = stable_test

    print(
        f"Base blend picked: {best_base_name} | "
        f"OOF={best_base_score:.6f} | "
        f"weights={[round(float(x), 4) for x in best_base_weights]}"
    )

    best_mix_alpha = 0.0
    best_mix_score = -1.0
    best_mix_oof = oof_best.copy()
    for alpha in list(np.linspace(0.0, 0.45, 10)):
        mix_oof = (1.0 - alpha) * oof_best + alpha * base_oof
        mix_pred = [idx_to_class[i] for i in np.argmax(mix_oof, axis=1)]
        mix_score = balanced_accuracy_score(y, mix_pred)
        if mix_score > best_mix_score:
            best_mix_score = mix_score
            best_mix_alpha = alpha
            best_mix_oof = mix_oof

    oof_best = best_mix_oof
    test_proba = (1.0 - best_mix_alpha) * test_proba + best_mix_alpha * base_test

    best_classwise_lam = np.zeros(len(classes), dtype=np.float64)
    best_classwise_score = -1.0
    if USE_CLASSWISE_STACK_BASE_BLEND:
        oof_for_classwise = oof_best.copy()
        test_for_classwise = test_proba.copy()
        for l0 in CLASSWISE_BLEND_GRID:
            for l1 in CLASSWISE_BLEND_GRID:
                for l2 in CLASSWISE_BLEND_GRID:
                    lam = np.array([l0, l1, l2], dtype=np.float64)
                    blend_oof = (1.0 - lam) * oof_for_classwise + lam * base_oof
                    blend_pred = [idx_to_class[i] for i in np.argmax(blend_oof, axis=1)]
                    sc = balanced_accuracy_score(y, blend_pred)
                    if sc > best_classwise_score:
                        best_classwise_score = sc
                        best_classwise_lam = lam.copy()
        if USE_ADAPTIVE_CLASSWISE_REFINE:
            lam0 = best_classwise_lam.copy()
            deltas = (
                np.arange(-CLASSWISE_FINE_DELTA_STEPS, CLASSWISE_FINE_DELTA_STEPS + 1, dtype=np.float64)
                * CLASSWISE_FINE_DELTA_STEP
            )
            for d0 in deltas:
                for d1 in deltas:
                    for d2 in deltas:
                        lam = np.clip(
                            lam0 + np.array([d0, d1, d2], dtype=np.float64), 0.0, 0.85
                        )
                        blend_oof = (1.0 - lam) * oof_for_classwise + lam * base_oof
                        blend_pred = [idx_to_class[i] for i in np.argmax(blend_oof, axis=1)]
                        sc = balanced_accuracy_score(y, blend_pred)
                        if sc > best_classwise_score:
                            best_classwise_score = sc
                            best_classwise_lam = lam.copy()
        oof_best = (1.0 - best_classwise_lam) * oof_for_classwise + best_classwise_lam * base_oof
        test_proba = (1.0 - best_classwise_lam) * test_for_classwise + best_classwise_lam * base_test
    else:
        best_classwise_score = balanced_accuracy_score(
            y, [idx_to_class[i] for i in np.argmax(oof_best, axis=1)]
        )

    oof_scale_base = oof_best.copy()
    best_class_scale = np.ones(len(classes), dtype=np.float64)
    best_class_scale_score = -1.0
    _scale_grid = np.linspace(0.80, 1.20, 11)
    for s0 in _scale_grid:
        for s1 in _scale_grid:
            for s2 in _scale_grid:
                scale = np.array([s0, s1, s2], dtype=np.float64)
                scaled_oof = oof_scale_base * scale
                scaled_pred = [idx_to_class[i] for i in np.argmax(scaled_oof, axis=1)]
                score = balanced_accuracy_score(y, scaled_pred)
                if score > best_class_scale_score:
                    best_class_scale_score = score
                    best_class_scale = scale.copy()
    if USE_ADAPTIVE_SCALE_REFINE:
        b0, b1, b2 = best_class_scale
        g0 = np.linspace(
            b0 - SCALE_FINE_HALF_SPAN, b0 + SCALE_FINE_HALF_SPAN, SCALE_FINE_N
        )
        g1 = np.linspace(
            b1 - SCALE_FINE_HALF_SPAN, b1 + SCALE_FINE_HALF_SPAN, SCALE_FINE_N
        )
        g2 = np.linspace(
            b2 - SCALE_FINE_HALF_SPAN, b2 + SCALE_FINE_HALF_SPAN, SCALE_FINE_N
        )
        for s0 in g0:
            for s1 in g1:
                for s2 in g2:
                    scale = np.clip(
                        np.array([s0, s1, s2], dtype=np.float64), 0.72, 1.28
                    )
                    scaled_oof = oof_scale_base * scale
                    scaled_pred = [
                        idx_to_class[i] for i in np.argmax(scaled_oof, axis=1)
                    ]
                    score = balanced_accuracy_score(y, scaled_pred)
                    if score > best_class_scale_score:
                        best_class_scale_score = score
                        best_class_scale = scale.copy()
    if REFINE_CLASS_SCALE:
        b0, b1, b2 = best_class_scale
        _fine = np.linspace(-0.05, 0.05, 11)
        for d0 in _fine:
            for d1 in _fine:
                for d2 in _fine:
                    scale = np.array(
                        [b0 + d0, b1 + d1, b2 + d2], dtype=np.float64
                    )
                    scale = np.clip(scale, 0.72, 1.28)
                    scaled_oof = oof_scale_base * scale
                    scaled_pred = [
                        idx_to_class[i] for i in np.argmax(scaled_oof, axis=1)
                    ]
                    score = balanced_accuracy_score(y, scaled_pred)
                    if score > best_class_scale_score:
                        best_class_scale_score = score
                        best_class_scale = scale.copy()

    oof_best = oof_scale_base * best_class_scale
    test_proba = test_proba * best_class_scale
    test_proba = test_proba / np.clip(test_proba.sum(axis=1, keepdims=True), 1e-12, None)

    threshold_offsets = np.zeros(len(classes), dtype=np.float64)
    threshold_score = -1.0
    if USE_BA_CLASS_THRESHOLD_SEARCH:
        threshold_offsets, threshold_score = optimize_class_threshold_offsets(
            oof_best, y, classes
        )
        oof_best = oof_best + threshold_offsets
        test_proba = test_proba + CLASS_THRESHOLD_TRANSFER_STRENGTH * threshold_offsets
        test_proba = test_proba / np.clip(test_proba.sum(axis=1, keepdims=True), 1e-12, None)

    cv_score = balanced_accuracy_score(y, [idx_to_class[i] for i in np.argmax(oof_best, axis=1)])
    print(f"OOF safety-mix alpha: {best_mix_alpha:.2f} | score={best_mix_score:.6f}")
    classwise_text = ", ".join(
        [f"{cls}={w:.2f}" for cls, w in zip(classes, best_classwise_lam)]
    )
    print(
        f"Classwise stack-base blend: {classwise_text} | score={best_classwise_score:.6f}"
    )
    class_scale_text = ", ".join([f"{cls}={s:.2f}" for cls, s in zip(classes, best_class_scale)])
    print(f"Best class probability scales: {class_scale_text} | score={best_class_scale_score:.6f}")
    if USE_BA_CLASS_THRESHOLD_SEARCH:
        th_text = ", ".join([f"{cls}={v:+.3f}" for cls, v in zip(classes, threshold_offsets)])
        print(f"Class threshold offsets: {th_text} | score={threshold_score:.6f} | transfer={CLASS_THRESHOLD_TRANSFER_STRENGTH}")
    print("\nCV balanced accuracy (OOF):", round(cv_score, 6))
    print("Fold equal-blend scores:", [round(s, 6) for s in fold_scores])
    print("Mean equal-blend score:", round(float(np.mean(fold_scores)), 6))
    print(f"Best OOF score from stacking search: {best_stack_score:.6f}")
    print(f"Best temperature used: {best_T}")
    bias_text = ", ".join(
        [f"{cls}={offset:+.3f}" for cls, offset in zip(classes, best_bias)]
    )
    print(f"Best class bias offsets: {bias_text}")
    print(f"Best OOF score after bias tuning: {best_bias_score:.6f}")
    print(f"Total bias combos checked: {bias_checks}")
    print(f"Phase run time: {(time.perf_counter() - run_start) / 60.0:.2f} minutes")

    oof_pred_labels = [idx_to_class[i] for i in np.argmax(oof_best, axis=1)]
    cm_counts = confusion_matrix(y, oof_pred_labels, labels=classes)
    cm_counts_df = pd.DataFrame(cm_counts, index=classes, columns=classes)
    row_sums = cm_counts.sum(axis=1, keepdims=True)
    row_sums = np.where(row_sums == 0, 1, row_sums)
    cm_row_pct_df = pd.DataFrame(
        np.round(cm_counts / row_sums, 4),
        index=classes,
        columns=classes,
    )

    return {
        "test_proba": test_proba,
        "test_after_temp": test_after_temp,
        "test_proba_dict": {
            "stack": test_after_temp,
            "cat": test_cat,
            "lgb": test_lgb,
            "xgb": test_xgb,
            "base_blend": base_test,
        },
        "cv_score": cv_score,
        "oof_bias_score": best_bias_score,
        "oof_pred": oof_pred_labels,
        "y_true": y,
        "cm_counts": cm_counts_df,
        "cm_row_pct": cm_row_pct_df,
    }


total_t0 = time.perf_counter()
print("Starting ensemble CV run...")
print(
    f"ORIGINAL_ROW_WEIGHT={ORIGINAL_ROW_WEIGHT} | "
    f"USE_PSEUDO_LABEL={USE_PSEUDO_LABEL} | "
    f"PSEUDO_CONF_MIN={PSEUDO_CONF_MIN}"
)

out1 = run_training_phase(TRAIN_DF_BASE.copy(), "phase1")

final_out = out1
test_proba = out1["test_proba"]

if USE_PSEUDO_LABEL:
    train_aug, pseudo_w_map = build_pseudo_augmented_train_curriculum(
        TRAIN_DF_BASE,
        test_df,
        out1["test_proba_dict"],
    )
    if len(train_aug) > len(TRAIN_DF_BASE):
        out2 = run_training_phase(train_aug, "phase2", pseudo_weight_map=pseudo_w_map)
        final_out = out2
        test_proba = out2["test_proba"]

print(f"\nTotal wall time: {(time.perf_counter() - total_t0) / 60.0:.2f} minutes")
print("\nOOF confusion matrix (counts):")
display(final_out["cm_counts"])
print("\nOOF confusion matrix (row-normalized):")
display(final_out["cm_row_pct"])


Starting ensemble CV run...
ORIGINAL_ROW_WEIGHT=0.35 | USE_PSEUDO_LABEL=False | PSEUDO_CONF_MIN=0.994

==================== phase1 ====================
Config: OVERNIGHT_MODE=False | seeds=3 | n_splits=3 | stacker_splits=5 | USE_CAT2=True | USE_HGB=True | USE_HGB2=False | USE_LGBM_STACKER=True | C_grid=15 | LGB_stack_grid=2 | T_grid=14
CV repeat schedule: [{'n_splits': 3, 'seeds': [42, 1337, 2026]}, {'n_splits': 4, 'seeds': [31415]}]

######## CV round 1/4 (seed=42, splits=3) ########

=== Seed 42 | Fold 1/3 started ===
Seed 42 Fold 1: training CatBoost...
0:	learn: 0.9494418	test: 0.8167563	best: 0.8167563 (0)	total: 1.09s	remaining: 12m 39s
100:	learn: 0.9769254	test: 0.9209418	best: 0.9209536 (96)	total: 1m 29s	remaining: 8m 50s
200:	learn: 0.9808419	test: 0.9285595	best: 0.9286041 (199)	total: 2m 56s	remaining: 7m 18s
300:	learn: 0.9848809	test: 0.9357572	best: 0.9357572 (300)	total: 4m 23s	remaining: 5m 48s
400:	learn: 0.9873638	test: 0.9422973	best: 0.9422973 (400)	total: 5m 46s	

C:\Users\ol1v3_7dwns5u\AppData\Roaming\Python\Python311\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
C:\Users\ol1v3_7dwns5u\AppData\Roaming\Python\Python311\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


Training until validation scores don't improve for 60 rounds
Did not meet early stopping. Best iteration is:
[220]	valid_0's multi_logloss: 0.0630629


C:\Users\ol1v3_7dwns5u\AppData\Roaming\Python\Python311\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
C:\Users\ol1v3_7dwns5u\AppData\Roaming\Python\Python311\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


Stacking LGBM li=0: finished meta-fold 2/5
Training until validation scores don't improve for 60 rounds
Did not meet early stopping. Best iteration is:
[220]	valid_0's multi_logloss: 0.064047


...

C:\Users\ol1v3_7dwns5u\AppData\Roaming\Python\Python311\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
C:\Users\ol1v3_7dwns5u\AppData\Roaming\Python\Python311\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


Stacking LGBM li=1: cross-fitted OOF balanced_accuracy = 0.971034
Best stacker: LogisticRegression C=0.85 | OOF=0.972924
Searching temperature scaling on stacker OOF...
Best temperature T=0.64 OOF balanced_accuracy=0.972924
Starting class-bias tuning...
Bias search progress: checked 200 combos
Bias search progress: checked 400 combos
Bias search progress: checked 600 combos
Bias search progress: checked 800 combos
Bias search progress: checked 1000 combos
Bias search progress: checked 1200 combos
Bias search progress: checked 1400 combos
Bias search progress: checked 1600 combos
Bias search progress: checked 1800 combos
Bias search progress: checked 2000 combos
Base blend picked: cat2_primary_0.36 | OOF=0.970103 | weights=[0.128, 0.128, 0.128, 0.36, 0.128, 0.128]
OOF safety-mix alpha: 0.00 | score=0.972953
Classwise stack-base blend: High=0.00, Low=0.55, Medium=0.00 | score=0.972961
Best class probability scales: High=1.00, Low=1.08, Medium=1.00 | score=0.972963
Class threshold offsets

,High,Low,Medium
High,20541,0,804
Low,2,374198,1581
Medium,4383,5145,233346



OOF confusion matrix (row-normalized):


,High,Low,Medium
High,0.9623,0.0000,0.0377
Low,0.0000,0.9958,0.0042
Medium,0.0180,0.0212,0.9608


In [6]:
test_proba_blend = test_proba.copy()
if ANCHOR_BLEND_WEIGHT > 0 and ANCHOR_SUB_PATH.exists():
    anchor_df = pd.read_csv(ANCHOR_SUB_PATH)[[ID_COL, TARGET]].copy()
    anchor_map = anchor_df.set_index(ID_COL)[TARGET]
    anchor_labels = test_df[ID_COL].map(anchor_map)
    if anchor_labels.isna().any():
        raise ValueError("Anchor blend file is missing ids from test set")
    anchor_idx = anchor_labels.map(class_to_idx).astype(int).values
    anchor_oh = np.zeros_like(test_proba_blend, dtype=np.float64)
    anchor_oh[np.arange(len(anchor_oh)), anchor_idx] = 1.0
    w = float(ANCHOR_BLEND_WEIGHT)
    test_proba_blend = (1.0 - w) * test_proba_blend + w * anchor_oh
    print(f"Anchor probability blend applied (weight={w})")
else:
    print("Anchor probability blend skipped (weight=0 or file missing)")

for path, wt in SUBMISSION_EXTRA_BLENDS:
    if wt <= 0:
        continue
    if not path.exists():
        print(f"Extra submission blend skipped (missing): {path}")
        continue
    sub_df = pd.read_csv(path)[[ID_COL, TARGET]].copy()
    amap = sub_df.set_index(ID_COL)[TARGET]
    alabels = test_df[ID_COL].map(amap)
    if alabels.isna().any():
        raise ValueError(f"Extra blend file missing test ids: {path}")
    aidx = alabels.map(class_to_idx).astype(int).values
    aoh = np.zeros_like(test_proba_blend, dtype=np.float64)
    aoh[np.arange(len(aoh)), aidx] = 1.0
    w = float(wt)
    test_proba_blend = (1.0 - w) * test_proba_blend + w * aoh
    print(f"Extra submission blend: {path.name} weight={w}")

test_conf = np.max(test_proba_blend, axis=1)
test_pred_idx = np.argmax(test_proba_blend, axis=1)
test_pred = pd.Series([idx_to_class[i] for i in test_pred_idx], index=test_df.index)

if USE_ANCHOR_FALLBACK and ANCHOR_SUB_PATH.exists():
    anchor_df = pd.read_csv(ANCHOR_SUB_PATH)
    anchor_df = anchor_df[[ID_COL, TARGET]].copy()
    anchor_map = anchor_df.set_index(ID_COL)[TARGET]
    anchor_labels = test_df[ID_COL].map(anchor_map)

    if anchor_labels.isna().any():
        raise ValueError("Anchor submission is missing ids from test set")

    use_anchor_mask = test_conf < FALLBACK_CONF_THRESHOLD
    test_pred.loc[use_anchor_mask] = anchor_labels.loc[use_anchor_mask]
    print(
        f"Anchor fallback applied to {int(use_anchor_mask.sum())} rows "
        f"(threshold={FALLBACK_CONF_THRESHOLD})"
    )
else:
    print("Anchor fallback disabled or anchor file missing")

_sample_cols = pd.read_csv(DATA_DIR / "sample_submission.csv", nrows=0).columns.tolist()
_sub_id_name, _sub_target_name = _sample_cols[0], _sample_cols[1]
submission = pd.DataFrame(
    {
        _sub_id_name: test_df[ID_COL].astype("int64").to_numpy(),
        _sub_target_name: test_pred.to_numpy(),
    }
)
assert len(submission) == len(test_df), "submission length must match test"
submission.to_csv(SUB_PATH, index=False, encoding="utf-8", lineterminator="\n")

print("Saved:", SUB_PATH.resolve())
print("Submission columns (must match sample):", list(submission.columns))
submission.head()


Anchor probability blend applied (weight=0.06)
Anchor fallback disabled or anchor file missing
Saved: C:\Users\ol1v3_7dwns5u\OneDrive\Desktop\ACTIVE\s6e4-predicting-irrigation-need\submission.csv
Submission columns (must match sample): ['id', 'Irrigation_Need']


,id,Irrigation_Need
0,630000,Low
1,630001,Low
2,630002,Low
3,630003,Low
4,630004,Low
